# 03 — Model training, independent calibration and untouched test
Model selection uses validation ROC-AUC. Calibration fits **only** on validation labels. Final metrics use the untouched test set. This is a demonstration on an education dataset, not a Dubai real-estate performance claim.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from lead_scoring_utils import *

import joblib
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, classification_report
from sklearn.isotonic import IsotonicRegression
train,val,test=load_split()
Xtr,Xv,Xte=map(prepare,(train,val,test))
ytr,yv,yte=[d['Converted'].to_numpy() for d in (train,val,test)]
models={
 'Logistic Regression':LogisticRegression(max_iter=2000,random_state=42),
 'Random Forest':RandomForestClassifier(n_estimators=150,max_depth=8,min_samples_leaf=8,n_jobs=-1,random_state=42),
 'XGBoost':XGBClassifier(n_estimators=180,max_depth=4,learning_rate=.05,subsample=.85,colsample_bytree=.85,tree_method='hist',n_jobs=2,random_state=42,eval_metric='logloss')
}
fitted={}
scores={}
for name,est in models.items():
    pipe=Pipeline([('preprocess',preprocessing()),('model',est)])
    pipe.fit(Xtr,ytr)
    p=pipe.predict_proba(Xv)[:,1]
    scores[name]=roc_auc_score(yv,p)
    fitted[name]=pipe
    print(name,'validation ROC-AUC',round(scores[name],4))
winner=max(scores,key=scores.get)
model=fitted[winner]
print('Selected on validation only:',winner)

Logistic Regression validation ROC-AUC 0.8576


Random Forest validation ROC-AUC 0.8647
XGBoost validation ROC-AUC 0.8728
Selected on validation only: XGBoost


In [2]:
# Calibration fitted on validation set, evaluated ONLY on held-out test.
p_val=model.predict_proba(Xv)[:,1]
calibrator=IsotonicRegression(out_of_bounds='clip').fit(p_val,yv)
p_test_raw=model.predict_proba(Xte)[:,1]
p_test=calibrator.predict(p_test_raw)
metrics={
 'selected_model':winner,
 'validation_auc':float(scores[winner]),
 'test_auc_raw':float(roc_auc_score(yte,p_test_raw)),
 'test_auc_calibrated':float(roc_auc_score(yte,p_test)),
 'test_average_precision':float(average_precision_score(yte,p_test)),
 'test_brier_calibrated':float(brier_score_loss(yte,p_test)),
 'test_lift_top_20pct':lift_at_k(yte,p_test,.2),
 'test_base_conversion':float(yte.mean()),
 'test_n':len(yte),
}
print(pd.Series(metrics))
print('\nTier conversion on held-out test:')
results=pd.DataFrame({'actual':yte,'probability':p_test,'tier':tiers(p_test)})
print(results.groupby('tier')['actual'].agg(['size','mean']).to_string())
joblib.dump(
    {'pipeline': model, 'calibrator': calibrator, 'feature_columns': FEATURES},
    BASE / 'models' / 'lead_scoring_pipeline.joblib'
)

pd.Series(metrics).to_json(
    BASE / 'data' / 'processed' / 'verified_metrics.json',
    indent=2
)

results.to_csv(
    BASE / 'data' / 'processed' / 'test_scored.csv',
    index=False
)

print('Saved model bundle, test scores and metrics.')

selected_model             XGBoost
validation_auc            0.872838
test_auc_raw              0.866398
test_auc_calibrated       0.865272
test_average_precision    0.797995
test_brier_calibrated     0.139682
test_lift_top_20pct        2.27807
test_base_conversion      0.385281
test_n                        1386
dtype: object

Tier conversion on held-out test:
      size      mean
tier                
Cold   714  0.117647
Hot    403  0.821340
Warm   269  0.442379


Saved model bundle, test scores and metrics.
